# Evaluating RAG with LangChain and Langfuse

RAG has two parts that can fail on their own. Retrieval can bring back the wrong documents, and the model can answer badly from good documents.

We do not write the evaluators here. Langfuse already has judges for both halves, and this lesson uses them.

This notebook keeps the vectors in memory. Nothing to install, nothing to start.

In [ ]:
%pip install -q "langchain>=1.0,<2" "langchain-openrouter>=0.1,<1" "langchain-openai>=1.6,<2" "langfuse>=4,<5" "pandas>=2.2,<4" "python-dotenv>=1.0,<2"

In [ ]:
import json
import os
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_openrouter import ChatOpenRouter
from langfuse import Evaluation, get_client, observe
from langfuse.langchain import CallbackHandler
from langchain_core.vectorstores import InMemoryVectorStore

load_dotenv()

langfuse = get_client()
langfuse_handler = CallbackHandler()
print("Langfuse connected:", langfuse.auth_check())

model = ChatOpenRouter(
    model=os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini"),
    app_title="PyData Amsterdam LLM Evaluation Tutorial",
)

# OpenRouter also serves embeddings and its API has the OpenAI shape,
# so we use the OpenAI client and only change the base URL. One API key for both.
embeddings = OpenAIEmbeddings(
    model=os.getenv("OPENROUTER_EMBEDDING_MODEL", "openai/text-embedding-3-small"),
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

## 1. The data

`dataset.json` holds the attendee guide documents and the test questions.

In [ ]:
dataset_candidates = [
    Path("dataset.json"),
    Path("03_rag/dataset.json"),
    Path("examples/03_rag/dataset.json"),
]
data = json.loads(next(p for p in dataset_candidates if p.exists()).read_text())

documents = [
    Document(page_content=d["text"], metadata={"doc_id": d["doc_id"], "title": d["title"]})
    for d in data["documents"]
]
examples = data["examples"]

print(f"{len(documents)} documents, {len(examples)} questions")
pd.DataFrame(data["documents"])[["doc_id", "title"]]

## 2. The prompt in Langfuse

The prompt takes the retrieved text as `{{context}}` and the question as `{{question}}`. It tells the model to answer only from the context, so the judges can later check whether it obeyed.

In [ ]:
RAG_PROMPT = "pydata-attendee-guide"

langfuse.create_prompt(
    name=RAG_PROMPT,
    type="text",
    prompt=(
        "You are a PyData Amsterdam assistant.\n"
        "Answer the question using only the context below.\n"
        "If the context does not answer the question, say that you do not know.\n\n"
        "Context:\n{{context}}\n\n"
        "Question: {{question}}"
    ),
    labels=["production"],
)

prompt = ChatPromptTemplate.from_template(
    langfuse.get_prompt(RAG_PROMPT).get_langchain_prompt()
)
answer_chain = prompt | model | StrOutputParser()

## 3. The vector store

`InMemoryVectorStore` comes with LangChain. Nothing to install and nothing to start, which is enough for ten documents.

The vectors live in this kernel. When the kernel stops they are gone, and there is no way to look inside them. `langchain_with_qdrant.ipynb` runs the same lesson on a real vector database.

In [ ]:
store = InMemoryVectorStore.from_documents(documents, embeddings)
print(f"{len(documents)} documents embedded in memory")

## 4. The RAG pipeline

A **retriever** searches the store and gives back the documents that are closest to the question. `k` says how many it returns.

`ask` does the three steps of RAG:

1. retrieve the documents,
2. join their text into one context,
3. ask the model to answer from that context.

`@observe` gives the trace a name, and both steps pass the Langfuse handler, so retrieval and generation show up as separate steps inside it.

`ask` returns three things: the answer, which documents were used, and their text. The judges in section 6 need that text.

In [ ]:
@observe(name="RAG Pipeline")
def ask(retriever, question: str) -> dict:
    found = retriever.invoke(question, config={"callbacks": [langfuse_handler]})
    context = "\n\n".join(f"[{d.metadata['doc_id']}] {d.page_content}" for d in found)

    answer = answer_chain.invoke(
        {"context": context, "question": question},
        config={"callbacks": [langfuse_handler]},
    )

    return {
        "answer": answer,
        "doc_ids": [d.metadata["doc_id"] for d in found],
        "retrieved_contexts": [d.page_content for d in found],
    }

In [ ]:
# Ask the store for the 3 documents closest to the question.
retriever = store.as_retriever(search_kwargs={"k": 3})

result = ask(retriever, "How do I connect to the wifi?")
langfuse.flush()

print("documents used:", result["doc_ids"])
print()
print(result["answer"])

## 5. The dataset in Langfuse

In [ ]:
timestamp = datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
dataset_name = f"pydata-rag-{timestamp}"

langfuse.create_dataset(name=dataset_name, description="Attendee guide questions.")
for example in examples:
    langfuse.create_dataset_item(dataset_name=dataset_name, **example)

dataset = langfuse.get_dataset(dataset_name)
print(f"{dataset.name} has {len(dataset.items)} items")

## 6. Our own evaluators

Langfuse also ships built-in judges, but attaching them from Python needs an API that Langfuse marks as unstable. So we write two ourselves, one for each half of RAG.

`retrieves_expected_documents` is plain Python. It compares the documents that came back with the ones the dataset expects. Cheap, fast, and it gives the same score every time.

`answer_is_grounded` is an LLM judge. It asks a model whether the context supports the answer. That is all a judge is: a prompt and a model. So its prompt lives in Langfuse, like every other prompt in this tutorial.

In [ ]:
JUDGE_PROMPT = "pydata-groundedness-judge"

langfuse.create_prompt(
    name=JUDGE_PROMPT,
    type="text",
    prompt=(
        "Does the context support every statement in the answer?\n"
        "Answer with one word, yes or no.\n\n"
        "Context:\n{{context}}\n\n"
        "Answer:\n{{answer}}"
    ),
    labels=["production"],
)

judge_chain = (
    ChatPromptTemplate.from_template(
        langfuse.get_prompt(JUDGE_PROMPT).get_langchain_prompt()
    )
    | model
    | StrOutputParser()
)


def retrieves_expected_documents(*, output, expected_output, **kwargs) -> Evaluation:
    expected = expected_output["doc_ids"]
    missing = [doc_id for doc_id in expected if doc_id not in output["doc_ids"]]

    return Evaluation(
        name="retrieves_expected_documents",
        value=(len(expected) - len(missing)) / len(expected),
        comment=f"did not retrieve: {', '.join(missing)}" if missing else "All expected documents retrieved.",
    )


def answer_is_grounded(*, output, **kwargs) -> Evaluation:
    verdict = judge_chain.invoke(
        {
            "context": "\n\n".join(output["retrieved_contexts"]),
            "answer": output["answer"],
        }
    )

    return Evaluation(
        name="answer_is_grounded",
        value=verdict.strip().casefold().startswith("yes"),
        comment=f"the judge said: {verdict.strip()}",
    )

## 7. Experiment: how many documents to retrieve

Same documents, same prompt, same model. Only `k` changes, which is how many documents the retriever returns.

`run_experiment` calls the task once for every question, then runs both evaluators on what it returned.

We write the two runs out in full instead of looping, so you can read each one on its own.

In [ ]:
retriever_k1 = store.as_retriever(search_kwargs={"k": 1})
retriever_k3 = store.as_retriever(search_kwargs={"k": 3})


def answer_with_1_document(*, item, **kwargs) -> dict:
    return ask(retriever_k1, item.input["question"])


def answer_with_3_documents(*, item, **kwargs) -> dict:
    return ask(retriever_k3, item.input["question"])


run_k1 = dataset.run_experiment(
    name="rag-k1",
    task=answer_with_1_document,
    evaluators=[retrieves_expected_documents, answer_is_grounded],
    max_concurrency=2,
)

run_k3 = dataset.run_experiment(
    name="rag-k3",
    task=answer_with_3_documents,
    evaluators=[retrieves_expected_documents, answer_is_grounded],
    max_concurrency=2,
)

langfuse.flush()

print("k=1", run_k1.dataset_run_url)
print("k=3", run_k3.dataset_run_url)

Read the two scores together. Low retrieval with a grounded answer means the model answered honestly from the wrong documents. Good retrieval with an ungrounded answer means the model invented something it was not given.

In [ ]:
rows = []

for name, run in [("k=1", run_k1), ("k=3", run_k3)]:
    for item_result in run.item_results:
        row = {"experiment": name, "question": item_result.item.input["question"]}
        for evaluation in item_result.evaluations:
            row[evaluation.name] = evaluation.value
            row[f"{evaluation.name}: why"] = evaluation.comment
        rows.append(row)

pd.DataFrame(rows).set_index(["experiment", "question"])